# Driver Bid Fare Recommender  (v3)

Recommends what a driver should bid, as a **band**, for `one_way` and `round_way`
bookings. 

## The accuracy work, specifically

Five changes, each of which earns or loses its place in the §17 comparison table
rather than being assumed to help:

1. **Optuna tuning per segment** (30 trials). Objective is median absolute percentage
   error on fare, not RMSE on the log target — that's what a driver experiences.
2. **A second target.** Since `fare ≈ km × rate`, taking logs gives
   `log(fare) = log(km) + log(rate)`. Modelling `log(rate)` directly hands the model
   the distance scaling for free and lets every split it makes be about *rate
   deviation* instead. Both targets are fitted and both go in the ensemble.
3. **A monotone-in-distance model.** Fare should never fall as distance rises.
   Constraining it makes long-trip extrapolation behave, which matters because the
   holdout's longest trips are exactly where an unconstrained tree invents things.
4. **Km-band rate cards.** Short trips earn far more per km than long ones. The v2
   rate card was keyed only on district, so it partly confounded distance with
   geography. v3 adds a distance-band rate card alongside it and blends the two.
5. **Recency weighting.** Rows are weighted by age with a one-year half-life, so a
   drifting market pulls the model toward recent prices.


## 1 · Setup

In [ ]:
!pip install -q lightgbm xgboost catboost optuna

In [ ]:
import warnings, json, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
import optuna
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

try:
    from catboost import CatBoostRegressor, Pool
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 200, "display.max_columns", 80)
sns.set_theme(style="whitegrid", palette="deep")
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__,
      "| optuna", optuna.__version__, "| catboost", HAS_CATBOOST)

## 2 · Configuration

In [ ]:
CFG = dict(
    # ---------- data ----------
    CSV_PATH   = "trips_data.csv",
    TARGET_RAW = "Driver Fare",

    # ---------- what to model ----------
    DROP_TRIP_TYPES = ("hourly", "airport_rental"),
    DROP_CAR_TYPES  = ("chander gari",),   # matched lowercased + substring, so variants go too
    SEGMENTS        = ("one_way", "round_way"),   # one model each

    # ---------- leakage switches ----------
    USE_PROMO           = False,
    USE_DRIVER_FEATURES = False,
    USE_ROUTE_RATE      = True,

    # ---------- cleaning ----------
    DROP_ZERO_FARE = True,
    PPK_MIN = 5.0, PPK_MAX = 400.0,     # hard implausibility gate (data-entry errors)
    PPK_TRIM_LOW  = 0.02,               # then a per-segment percentile trim on BDT/km:
    PPK_TRIM_HIGH = 0.98,               # bottom 2% and top 2% are dropped from the project

    # ---------- split ----------
    SPLIT_MODE = "time",   # "time" (recommended) | "random"
    TEST_SIZE  = 0.20,     # holdout, scored once
    CALIB_FRAC = 0.20,     # carved out of the 80%: 64% fit / 16% calib / 20% holdout

    # ---------- tuning ----------
    TUNE_TRIALS   = 30,    # Optuna trials per segment
    TUNE_VAL_FRAC = 0.20,  # slice of the fit set used for tuning + early stopping

    # ---------- modelling ----------
    BAND_ALPHA            = 0.20,   # 80% band
    RECENCY_HALFLIFE_DAYS = 365,    # sample weight half-life; None to disable
    MONOTONE_KM           = True,   # fit a fare-non-decreasing-in-distance member

    RANDOM_STATE = 42,
)
SEED = CFG["RANDOM_STATE"]

DHAKA   = (23.8103, 90.4125)
BD_BBOX = dict(lat=(20.0, 27.0), lon=(87.0, 93.0))

print("segments      :", CFG["SEGMENTS"])
print("dropping      :", CFG["DROP_TRIP_TYPES"], "+ car types", CFG["DROP_CAR_TYPES"])
print("ppk trim      : keep the middle "
      f"{100*(CFG['PPK_TRIM_HIGH']-CFG['PPK_TRIM_LOW']):.0f}% of BDT/km, per segment")
print("tuning        :", CFG["TUNE_TRIALS"], "Optuna trials per segment")

## 3 · Load the raw CSV

In [ ]:
try:
    from google.colab import files
    up = files.upload()                 # pick trips_data.csv
    CFG["CSV_PATH"] = list(up.keys())[0]
except Exception:
    pass                                # not on Colab, or file already present

raw = pd.read_csv(CFG["CSV_PATH"], low_memory=False)
print("raw shape:", raw.shape)
raw.head(3)

## 4 · Label normalisation maps

The same place shows up as `Dhaka`, `ঢাকা`, `Dhaka Division`, `ঢাকা বিভাগ`, `ダッカ`
and even `ঢাকা-১১৮৮০`. Left alone, a tree model treats these as five unrelated
locations and splits its data five ways.

In [ ]:
DIVISION_MAP = {
    "dhaka": "Dhaka", "ঢাকা": "Dhaka", "dhaka division": "Dhaka", "ঢাকা বিভাগ": "Dhaka",
    "ダッカ": "Dhaka", "ঢাকা-১১৮৮০": "Dhaka", "default pickup division": np.nan,
    "chattogram": "Chattogram", "chittagong": "Chattogram", "চট্টগ্রাম": "Chattogram",
    "chittagong division": "Chattogram", "চট্টগ্রাম বিভাগ": "Chattogram",
    "chattogram division": "Chattogram",
    "mymensingh": "Mymensingh", "ময়মনসিংহ": "Mymensingh", "ময়মনসিংহ বিভাগ": "Mymensingh",
    "rajshahi": "Rajshahi", "রাজশাহী": "Rajshahi", "rajshahi division": "Rajshahi",
    "khulna": "Khulna", "খুলনা": "Khulna", "khulna division": "Khulna",
    "sylhet": "Sylhet", "সিলেট": "Sylhet", "sylhet division": "Sylhet",
    "rangpur": "Rangpur", "রংপুর": "Rangpur", "rangpur division": "Rangpur",
    "barisal": "Barishal", "barishal": "Barishal", "বরিশাল": "Barishal",
    "バリサル": "Barishal", "barishal division": "Barishal",
}

DISTRICT_MAP = {
    "ঢাকা": "Dhaka", "চট্টগ্রাম": "Chattogram", "chittagong": "Chattogram",
    "গাজীপুর": "Gazipur", "নারায়ণগঞ্জ": "Narayanganj", "কুমিল্লা": "Cumilla",
    "comilla": "Cumilla", "ময়মনসিংহ": "Mymensingh", "টাঙ্গাইল": "Tangail",
    "চাঁদপুর": "Chandpur", "কিশোরগঞ্জ": "Kishoreganj", "সিলেট": "Sylhet",
    "খুলনা": "Khulna", "রাজশাহী": "Rajshahi", "রংপুর": "Rangpur", "বরিশাল": "Barishal",
    "noakhali": "Noakhali", "feni": "Feni", "bogra": "Bogura", "bogura": "Bogura",
    "jessore": "Jashore", "jashore": "Jashore", "barisal": "Barishal",
}

AC_MAP = {"ac": "AC", "এসি": "AC", "dual-ac": "Dual-AC", "ডুয়েল এসি": "Dual-AC",
          "dual ac": "Dual-AC", "non-ac": "Non-AC", "non ac": "Non-AC", "0": "Unknown"}

# 640 raw model names -> ~25 families, matched by keyword (first match wins)
CAR_MODEL_KEYWORDS = [
    ("hiace", "HiAce"), ("hi-ace", "HiAce"), ("hi ace", "HiAce"),
    ("noah", "Noah"), ("voxy", "Voxy"), ("esquire", "Esquire"),
    ("axio", "Axio"), ("fielder", "Fielder"), ("corolla", "Corolla"),
    ("allion", "Allion"), ("premio", "Premio"), ("probox", "Probox"),
    ("belta", "Belta"), ("vitz", "Vitz"), ("aqua", "Aqua"), ("prius", "Prius"),
    ("carina", "Carina"), ("sienta", "Sienta"), ("rush", "Rush"), ("harrier", "Harrier"),
    ("crown", "Crown"), ("micro", "Microbus"), ("coaster", "Coaster"),
    ("pajero", "Pajero"), ("cr-v", "CRV"), ("crv", "CRV"), ("x-trail", "XTrail"),
    ("hybrid", "OtherHybrid"),
]
print("maps loaded")

## 5 · Parsing helpers

In [ ]:
def to_num(s):
    ''''3,000' -> 3000.0 ; anything unparseable -> NaN.'''
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(),
                         errors="coerce")


def parse_dt(s):

    out = pd.to_datetime(s, format="%B %d, %Y, %I:%M %p", errors="coerce")
    miss = out.isna() & s.notna()
    if miss.any():
        out.loc[miss] = pd.to_datetime(s[miss], errors="coerce", format="mixed")
    return out


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def bearing_deg(lat1, lon1, lat2, lon2):
    '''Direction of travel. Proxies 'which highway corridor' better than raw lat/long.'''
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(lon2 - lon1)
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def norm_label(s, mapping):
    mapped = s.astype(str).str.strip().str.lower().map(mapping)
    out = mapped.fillna(s.astype(str).str.strip().str.title())
    return out.replace({"Nan": np.nan, "None": np.nan, "": np.nan})


def group_car_model(s):
    low = s.astype(str).str.lower()
    out = pd.Series("Other", index=s.index, dtype=object)
    for kw, label in CAR_MODEL_KEYWORDS:
        hit = low.str.contains(kw, regex=False, na=False) & (out == "Other")
        out.loc[hit] = label
    out.loc[s.isna()] = "Unknown"
    return out


def clean_coord(lat, lon):
    '''(0,0) placeholders and out-of-Bangladesh junk -> NaN. Trees handle NaN natively;
    dropping the row would throw away a perfectly good fare/distance observation.'''
    lat = pd.to_numeric(lat, errors="coerce")
    lon = pd.to_numeric(lon, errors="coerce")
    ok = lat.between(*BD_BBOX["lat"]) & lon.between(*BD_BBOX["lon"])
    return lat.where(ok), lon.where(ok)

## 6 · `standardize()` — raw columns → typed, canonical columns

Unchanged from v2 and deliberately drops no rows, so the identical function runs on a
single incoming booking at serving time. `High Demand` and `No Of Bids` are still absent
by design.

In [ ]:
def standardize(raw):
    d = pd.DataFrame(index=raw.index)
    g = lambda c: raw[c] if c in raw.columns else pd.Series(np.nan, index=raw.index, dtype=object)

    # target + billing units
    d["fare"]  = to_num(g(CFG["TARGET_RAW"]))
    d["km"]    = to_num(g("Total Km"))
    d["hours"] = pd.to_numeric(g("Hours"), errors="coerce").fillna(0.0).clip(0, 24)

    # booking attributes
    # normalise "Airport Rental" / "airport-rental" / "airport_rental" to one token
    d["trip_type"] = (g("Trip Type").astype(str).str.strip().str.lower()
                        .str.replace(r"[\s\-]+", "_", regex=True))
    d["car_type"]  = g("Car Type").astype(str).str.strip()
    d["seats"]     = pd.to_numeric(g("No Of Seat").astype(str).str.extract(r"(\d+)")[0],
                                   errors="coerce")

    # timestamps
    d["pickup_dt"]  = parse_dt(g("Pickup Date Time"))
    d["created_dt"] = parse_dt(g("Created At"))
    d["return_dt"]  = parse_dt(g("Return Date Time"))

    # geography
    d["pickup_division"]  = norm_label(g("Pickup Division"),  DIVISION_MAP)
    d["dropoff_division"] = norm_label(g("Dropoff Division"), DIVISION_MAP)
    d["pickup_district"]  = norm_label(g("Pickup District"),  DISTRICT_MAP)
    d["dropoff_district"] = norm_label(g("Dropoff District"), DISTRICT_MAP)
    d["pickup_lat"],  d["pickup_long"]  = clean_coord(g("Pickup Lat"),  g("Pickup Long"))
    d["dropoff_lat"], d["dropoff_long"] = clean_coord(g("Dropoff Lat"), g("Dropoff Long"))

    # driver + vehicle
    ac = norm_label(g("Driver Car Ac Status"), AC_MAP).fillna("Unknown")
    d["ac_status"]         = ac.where(ac.isin(["AC", "Dual-AC", "Non-AC"]), "Unknown")
    my                     = pd.to_numeric(g("Driver Car Model Year"), errors="coerce")
    d["car_model_year"]    = my.where(my.between(1980, 2027))
    d["car_model_group"]   = group_car_model(g("Driver Car Model Name"))
    d["driver_rating"]     = pd.to_numeric(g("Driver Rating"), errors="coerce").replace(0, np.nan)
    d["driver_trips"]      = pd.to_numeric(g("Driver No Of Trips"), errors="coerce")
    d["driver_car_rating"] = pd.to_numeric(g("Driver Car Rating"), errors="coerce").replace(0, np.nan)

 
    d["promo_amount_raw"]   = pd.to_numeric(g("Promo Code Amount"), errors="coerce")
    d["promo_discount_raw"] = pd.to_numeric(g("Promo Discount Amount (BDT)"), errors="coerce")

    # bookkeeping
    d["trip_status"] = g("Trip Status").astype(str).str.strip().str.lower()
    d["booking_id"]  = g("Booking ID")
    return d


std = standardize(raw)
print(std.shape)
assert not {"high_demand", "n_bids_raw"} & set(std.columns), "removed columns leaked back in"
std.head(3)

## 7 · Row filtering — changes 1 and 3

Two new rules on top of v2.

**`Chander Gari` is dropped.** Matched on a lowercased substring, so `chander gari`,
`Chander Gari` and `Chander-Gari` all go. It will vanish from the web app's dropdown
automatically, because the app reads its car list from the saved model.

**Then a per-segment percentile trim on ৳/km.** The `PPK_MIN`/`PPK_MAX` gate above it
catches outright data errors; this catches the merely weird — a 400 km trip that paid
৳8/km, a 12 km trip that paid ৳380/km. Both are real rows, but they are not the kind of
booking the recommender will be asked about, and a squared-error learner will distort a
lot of ordinary predictions trying to reach them.

Two details worth understanding:

* **Per segment, not global.** Round trips bill both legs, so their ৳/km distribution
  sits well below one-way's. A single global threshold would cut mostly round trips at
  the bottom and mostly one-way at the top.
* **Thresholds are computed on the full cleaned set**, before the split. That's a mild
  form of using test-set information — the *cut points* depend on rows you'll later
  score. It's standard practice for cleaning rules and the effect is tiny, but if you
  want it airtight, move this block inside `fit_segment()` and compute the quantiles on
  the training rows only.

> You asked for these rows to be dropped from the project entirely, so they are. If you
> ever want to know how the model does on them, `filter_rows` returns them as
> `trimmed_rows` — scoring that frame separately would tell you how far off it goes on
> unusual pricing, without those rows influencing training.

In [ ]:
def filter_rows(d, cfg=CFG):
    n0, rep = len(d), {}
    rep["trip_type_counts_before"] = d["trip_type"].value_counts().to_dict()

    # ---- keep only the distance-priced trip types ------------------------------
    n = len(d); d = d[~d["trip_type"].isin(cfg["DROP_TRIP_TYPES"])]
    rep["dropped_trip_types"] = n - len(d)

   
    ct = d["car_type"].astype(str).str.strip().str.lower()
    bad = pd.Series(False, index=d.index)
    for token in cfg["DROP_CAR_TYPES"]:
        bad |= ct.str.contains(token.strip().lower(), regex=False, na=False)
    rep["dropped_car_types"] = int(bad.sum())
    rep["dropped_car_type_labels"] = d.loc[bad, "car_type"].value_counts().to_dict()
    d = d[~bad]

    n = len(d); d = d[d["trip_status"].isin(["completed", "trip completed"])]
    rep["not_completed"] = n - len(d)

    n = len(d); d = d[d["pickup_dt"].notna()]
    rep["unparseable_pickup_dt"] = n - len(d)

    if cfg["DROP_ZERO_FARE"]:
        n = len(d); d = d[d["fare"].notna() & (d["fare"] > 0)]
        rep["zero_or_null_fare"] = n - len(d)

    n = len(d); d = d[d["km"].fillna(0) > 0]
    rep["missing_or_zero_km"] = n - len(d)

    n = len(d); d = d.drop_duplicates(subset=["booking_id"])
    rep["duplicate_booking_id"] = n - len(d)

    # ---- hard implausibility gate ---------------------------------------------
    ppk = d["fare"] / d["km"]
    n = len(d)
    d = d[~(ppk.lt(cfg["PPK_MIN"]) | ppk.gt(cfg["PPK_MAX"])).fillna(False)]
    rep["implausible_fare_per_km"] = n - len(d)

  
    ppk = d["fare"] / d["km"]
    keep_mask = pd.Series(True, index=d.index)
    bounds = {}
    for tt, g in d.groupby("trip_type"):
        p = ppk.loc[g.index]
        lo = p.quantile(cfg["PPK_TRIM_LOW"])
        hi = p.quantile(cfg["PPK_TRIM_HIGH"])
        bounds[tt] = (round(float(lo), 1), round(float(hi), 1))
        keep_mask.loc[g.index] = p.between(lo, hi)
    rep["ppk_trim_bounds_BDT_per_km"] = bounds
    trimmed = d[~keep_mask].copy()
    rep["ppk_percentile_trim"] = int((~keep_mask).sum())
    d = d[keep_mask]

    rep["rows_in"], rep["rows_out"] = n0, len(d)
    rep["retention_%"] = round(100 * len(d) / n0, 2)
    rep["trip_type_counts_after"] = d["trip_type"].value_counts().to_dict()
    return d.reset_index(drop=True), trimmed.reset_index(drop=True), rep


clean, trimmed_rows, report = filter_rows(std)
for k, v in report.items():
    print(f"{k:30s} {v}")
print(f"\ntrimmed aside (not used anywhere): {len(trimmed_rows)} rows")

## 8 · Feature engineering

Unchanged from v2. The per-segment pruning in §12 removes whatever turns out to be
constant inside a segment — `is_round` in both, the return-gap features in one-way — so
there's no need to maintain two versions of this function.

In [ ]:
def add_features(df):
    d = df.copy()

    # ---------- size / distance -------------------------------------------------
    d["log_km"]    = np.log1p(d["km"])

    d["haversine_km"]     = haversine_km(d["pickup_lat"], d["pickup_long"],
                                         d["dropoff_lat"], d["dropoff_long"])
    d["log_haversine_km"] = np.log1p(d["haversine_km"])
    # >1 means a winding route; ~2.7 on round trips because Total Km covers both legs
    d["detour_ratio"] = (d["km"] / d["haversine_km"].replace(0, np.nan)).clip(0, 20)
    d["bearing"] = bearing_deg(d["pickup_lat"], d["pickup_long"],
                               d["dropoff_lat"], d["dropoff_long"])

    d["is_round"]   = d["trip_type"].eq("round_way").astype(int)
    d["km_per_leg"] = np.where(d["is_round"] == 1, d["km"] / 2, d["km"])

    # ---------- geography -------------------------------------------------------
    d["pickup_dist_from_dhaka"]  = haversine_km(d["pickup_lat"], d["pickup_long"], *DHAKA)
    d["dropoff_dist_from_dhaka"] = haversine_km(d["dropoff_lat"], d["dropoff_long"], *DHAKA)
    d["lat_diff"]        = d["dropoff_lat"]  - d["pickup_lat"]
    d["long_diff"]       = d["dropoff_long"] - d["pickup_long"]
    d["same_district"]   = (d["pickup_district"] == d["dropoff_district"]).astype(int)
    d["same_division"]   = (d["pickup_division"] == d["dropoff_division"]).astype(int)
    d["pickup_is_dhaka"] = d["pickup_district"].eq("Dhaka").astype(int)

    # ---------- time ------------------------------------------------------------
    p = d["pickup_dt"]
    d["pickup_hour"]  = p.dt.hour
    d["pickup_dow"]   = p.dt.dayofweek
    d["pickup_month"] = p.dt.month
    d["pickup_year"]  = p.dt.year
    d["pickup_doy"]   = p.dt.dayofyear
    d["is_weekend"]       = p.dt.dayofweek.isin([4, 5]).astype(int)   # Fri/Sat weekend in BD
    d["is_night"]         = p.dt.hour.isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
    d["is_early_morning"] = p.dt.hour.between(4, 7).astype(int)
    for col, period in [("pickup_hour", 24), ("pickup_dow", 7), ("pickup_month", 12)]:
        d[f"{col}_sin"] = np.sin(2 * np.pi * d[col] / period)
        d[f"{col}_cos"] = np.cos(2 * np.pi * d[col] / period)
    d["days_since_start"] = (p - p.min()).dt.total_seconds() / 86400   # slow market drift

    lead = (p - d["created_dt"]).dt.total_seconds() / 3600
    d["lead_hours"]     = lead.where(lead.between(0, 24 * 120))
    d["log_lead_hours"] = np.log1p(d["lead_hours"])
    d["is_immediate"]   = (d["lead_hours"].fillna(99) < 2).astype(int)  # urgency -> pricing power

    gap = (d["return_dt"] - p).dt.total_seconds() / 3600
    d["return_gap_hours"] = gap.where(gap.between(0, 24 * 60))   # round_way only
    d["return_gap_days"]  = np.ceil(d["return_gap_hours"] / 24)
    d["is_multiday"]      = (d["return_gap_hours"] > 20).astype(int)

    # ---------- driver / vehicle ------------------------------------------------
    d["car_age"]          = (d["pickup_year"] - d["car_model_year"]).where(lambda s: s.between(0, 45))
    d["log_driver_trips"] = np.log1p(d["driver_trips"])
    d["driver_is_new"]    = (d["driver_trips"].fillna(0) < 3).astype(int)

    # ---------- commercial ------------------------------------------------------

    d["promo_amount"]   = d["promo_amount_raw"].fillna(0)
    d["promo_discount"] = d["promo_discount_raw"].fillna(0)
    return d


feat = add_features(clean)
print("engineered shape:", feat.shape)

### Feature lists

Same base list as v2. `ROUTE_COLS` has grown: the rate card now carries a distance-band
component and a car-type multiplier alongside the district hierarchy.

In [ ]:
NUM_BASE = [
    "km", "log_km", "km_per_leg",
    "haversine_km", "log_haversine_km", "detour_ratio", "bearing",
    "seats", "pickup_lat", "pickup_long", "dropoff_lat", "dropoff_long",
    "lat_diff", "long_diff", "pickup_dist_from_dhaka", "dropoff_dist_from_dhaka",
    "same_district", "same_division", "pickup_is_dhaka",
    "pickup_hour", "pickup_dow", "pickup_month", "pickup_year", "pickup_doy",
    "is_weekend", "is_night", "is_early_morning",
    "pickup_hour_sin", "pickup_hour_cos", "pickup_dow_sin", "pickup_dow_cos",
    "pickup_month_sin", "pickup_month_cos", "days_since_start",
    "lead_hours", "log_lead_hours", "is_immediate",
    "return_gap_hours", "return_gap_days", "is_multiday", "is_round",
]
NUM_DRIVER = ["driver_rating", "driver_trips", "log_driver_trips",
              "driver_car_rating", "car_age", "car_model_year", "driver_is_new"]
NUM_PROMO  = ["promo_amount", "promo_discount"]

ROUTE_COLS = [
    "route_ppk", "route_ppk_n", "baseline_fare", "log_baseline_fare",
    "kmband_ppk", "kmband_baseline_fare",            # distance-band rate card
    "car_rate_mult",                                 # car type's rate multiplier
    "blend_baseline_fare", "log_blend_baseline_fare",# geometric blend of the two
    "pickup_district_volume",
]

# distance features the monotone member is constrained on
MONO_UP = ["km", "log_km", "km_per_leg", "haversine_km", "log_haversine_km",
           "baseline_fare", "log_baseline_fare",
           "blend_baseline_fare", "log_blend_baseline_fare"]

CAT_BASE   = ["car_type", "pickup_division", "dropoff_division",
              "pickup_district", "dropoff_district"]      
CAT_DRIVER = ["ac_status", "car_model_group"]


def feature_lists(cfg=CFG):
    num, cat = list(NUM_BASE), list(CAT_BASE)
    if cfg["USE_DRIVER_FEATURES"]: num += NUM_DRIVER; cat += CAT_DRIVER
    if cfg["USE_PROMO"]:           num += NUM_PROMO
    if cfg["USE_ROUTE_RATE"]:      num += ROUTE_COLS
    return num, cat


NUM_COLS, CAT_COLS = feature_lists()
feat = add_features(clean)
print(f"{len(NUM_COLS)} numeric + {len(CAT_COLS)} categorical features before pruning")
print("\nrows per segment:")
print(feat.trip_type.value_counts().to_string())

## 9 · EDA — how different are the two segments?

This is the evidence for or against change 2. If the two segments price alike, separate
models mostly cost you data. If they price differently — different ৳/km level, different
slope against distance, different spread — separate models are the right call.

In [ ]:
seg = feat.assign(ppk=feat.fare / feat.km).groupby("trip_type").apply(
    lambda g: pd.Series({
        "n": len(g),
        "share_%": 0.0,
        "median_fare": g.fare.median(),
        "median_km": g.km.median(),
        "median_BDT_per_km": g.ppk.median(),
        "IQR_BDT_per_km": g.ppk.quantile(.75) - g.ppk.quantile(.25),
        "CV_of_ppk": g.ppk.std() / g.ppk.mean(),
        "pct_with_return": 100 * g.return_dt.notna().mean(),
    }), include_groups=False)
seg["share_%"] = (100 * seg.n / seg.n.sum()).round(1)
print(seg.round(2).to_string())

small = seg[seg.n < 1500]
if len(small):
    print("\nCAUTION: these segments have under 1,500 rows, which is thin for a dedicated")
    print("model with 30 tuning trials. Check §17 — if the segment model loses to the")
    print("global baseline, that is the data volume talking:", list(small.index))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))

for tt, g in feat.groupby("trip_type"):
    ax[0].scatter(g.km, g.fare, s=5, alpha=.25, label=tt)
ax[0].set(xlabel="Total Km", ylabel="Driver Fare (BDT)",
          title="Fare vs distance — different slopes?")
ax[0].legend()

for tt, g in feat.groupby("trip_type"):
    sns.kdeplot(x=(g.fare / g.km), ax=ax[1], label=tt, fill=True, alpha=.25, clip=(0, 150))
ax[1].set(xlabel="BDT per km", title="Rate distribution after the percentile trim")
ax[1].legend()

band = feat.assign(km_band=pd.qcut(feat.km, 8, duplicates="drop"),
                   ppk=feat.fare / feat.km)
piv = band.pivot_table(index="km_band", columns="trip_type", values="ppk",
                       aggfunc="median", observed=True)
piv.plot(marker="o", ax=ax[2])
ax[2].set(xlabel="distance band", ylabel="median BDT per km",
          title="Rate falls with distance — hence km-band rate cards")
ax[2].tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

## 10 · Correlation with `Driver Fare`, per segment

Same three measures as v2 (Pearson, Spearman, Pearson against the log target for
numerics; correlation ratio η for categoricals), but computed **within each segment**.
Correlations pooled across segments partly measure the difference *between* segments
rather than anything usable inside one.

`route_*` columns are excluded — they are built from the target, so a high correlation
there is arithmetic, not information.

In [ ]:
def correlation_ratio(categories, values):
    '''Eta: sqrt(between-group variance / total variance). 0..1, categorical -> numeric.'''
    t = pd.DataFrame({"c": categories.astype("string"), "y": values}).dropna()
    if len(t) < 50 or t["c"].nunique() < 2:
        return np.nan
    grand = t["y"].mean()
    g = t.groupby("c")["y"]
    ss_between = (g.size() * (g.mean() - grand) ** 2).sum()
    ss_total   = ((t["y"] - grand) ** 2).sum()
    return float(np.sqrt(ss_between / ss_total)) if ss_total > 0 else np.nan


def numeric_correlations(d, target="fare", exclude=()):
    y, ylog = d[target], np.log1p(d[target])
    cols = [c for c in d.select_dtypes(include=[np.number]).columns
            if c != target and c not in exclude]
    rows = []
    for c in cols:
        s = d[c]
        m = s.notna() & y.notna()
        if m.sum() < 50 or s[m].nunique() < 2:
            continue
        rows.append(dict(
            feature      = c,
            n            = int(m.sum()),
            missing_pct  = round(100 * (1 - m.mean()), 1),
            pearson      = s[m].corr(y[m]),
            spearman     = s[m].corr(y[m], method="spearman"),
            pearson_log  = s[m].corr(ylog[m]),
        ))
    out = pd.DataFrame(rows)
    return out.reindex(out.spearman.abs().sort_values(ascending=False).index).reset_index(drop=True)


def categorical_correlations(d, cats, target="fare"):
    rows = [dict(feature=c,
                 n_levels=int(d[c].astype("string").nunique()),
                 missing_pct=round(100 * d[c].isna().mean(), 1),
                 eta=correlation_ratio(d[c], d[target]),
                 eta_log=correlation_ratio(d[c], np.log1p(d[target])))
            for c in cats if c in d.columns]
    return pd.DataFrame(rows).sort_values("eta_log", ascending=False).reset_index(drop=True)

In [ ]:
for tt in CFG["SEGMENTS"]:
    d = feat[feat.trip_type == tt]
    if len(d) < 100:
        print(f"\n=== {tt}: only {len(d)} rows, skipping ==="); continue
    print(f"\n{'='*74}\n=== {tt}  (n = {len(d):,})\n{'='*74}")
    nc = numeric_correlations(d, exclude=ROUTE_COLS + ["is_round"])
    print(nc.head(15).round(3).to_string(index=False))
    cc = categorical_correlations(d, CAT_BASE + CAT_DRIVER)
    print("\ncategoricals:")
    print(cc.round(3).to_string(index=False))

## 11 · Splitting, weighting and targets

### The split, per segment

80% development / 20% holdout, then the development part is divided again into a fit set
and a calibration set. Percentages are of each segment, so both models get the same
proportions regardless of how lopsided the segment sizes are.

```
        |<---------------- 80% ---------------->|<---- 20% ---->|
        |<----- fit 64% ----->|<-- calib 16% -->|   holdout     |
             |<- tune 13% ->|
             (carved from the fit set for Optuna + early stopping)
```

Four slices, four distinct jobs, and none of them does two jobs at once:

| Slice | Used for |
|---|---|
| `fit` | training the base models |
| `tune` | Optuna trials and early stopping (carved out of `fit`) |
| `calib` | ensemble weights and the conformal band correction |
| `holdout` | scored once, at the end |

### Recency weighting

Fares drift. Rows are weighted by `0.5 ** (age_in_days / 365)`, so a booking from three
years ago counts about one-eighth as much as last month's. Set
`RECENCY_HALFLIFE_DAYS = None` to switch it off; the §17 table will tell you whether it
helped.

### Two targets

`log1p(fare)` as before, plus `log(fare / km)`. The second is the interesting one:
because `log(fare) = log(km) + log(rate)`, predicting the rate hands the model the
distance scaling for free, so every split it spends goes on explaining rate *deviation*
rather than re-learning that longer trips cost more. Both are fitted; the ensemble
decides how much to trust each.

In [ ]:
def split_frames(d, cfg=CFG):
    '''Return fit / tune / calib / holdout for one segment.'''
    n = len(d)
    if cfg["SPLIT_MODE"] == "time":
        d = d.sort_values("pickup_dt").reset_index(drop=True)
        cut = int(round(n * (1 - cfg["TEST_SIZE"])))
        dev, te = d.iloc[:cut], d.iloc[cut:]
        c2 = int(round(len(dev) * (1 - cfg["CALIB_FRAC"])))
        fit_all, ca = dev.iloc[:c2], dev.iloc[c2:]
        c3 = int(round(len(fit_all) * (1 - cfg["TUNE_VAL_FRAC"])))
        fi, tu = fit_all.iloc[:c3], fit_all.iloc[c3:]
    elif cfg["SPLIT_MODE"] == "random":
        dev, te = train_test_split(d, test_size=cfg["TEST_SIZE"], random_state=SEED)
        fit_all, ca = train_test_split(dev, test_size=cfg["CALIB_FRAC"], random_state=SEED)
        fi, tu = train_test_split(fit_all, test_size=cfg["TUNE_VAL_FRAC"], random_state=SEED)
    else:
        raise ValueError("SPLIT_MODE must be 'time' or 'random'")
    r = lambda f: f.reset_index(drop=True)
    return r(fi), r(tu), r(ca), r(te), r(fit_all)


def recency_weights(d, halflife=None, ref=None):
    if not halflife:
        return None
    ref = ref if ref is not None else d["pickup_dt"].max()
    age_days = (ref - d["pickup_dt"]).dt.total_seconds() / 86400
    return np.power(0.5, np.clip(age_days, 0, None) / halflife).to_numpy()


# ---- the two targets -------------------------------------------------------
TARGETS = {
    "logfare": dict(
        label = "log1p(fare)",
        fwd   = lambda d: np.log1p(d["fare"].to_numpy(dtype="float64")),
        inv   = lambda z, d: np.expm1(np.asarray(z, dtype="float64")),
    ),
    "logppk": dict(
        label = "log(fare per km)",
        fwd   = lambda d: np.log(d["fare"].to_numpy(dtype="float64")
                                 / d["km"].to_numpy(dtype="float64")),
        inv   = lambda z, d: np.exp(np.asarray(z, dtype="float64"))
                             * d["km"].to_numpy(dtype="float64"),
    ),
}


def metrics(y_true, y_pred, name=""):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.clip(np.asarray(y_pred, dtype="float64"), 1, None)
    ape = np.abs(y_true - y_pred) / np.clip(y_true, 1, None)
    return dict(model=name, n=len(y_true),
                R2=r2_score(y_true, y_pred),
                MAE=mean_absolute_error(y_true, y_pred),
                RMSE=math.sqrt(mean_squared_error(y_true, y_pred)),
                MAPE=100 * ape.mean(), MedAPE=100 * np.median(ape),
                within_10pct=100 * (ape <= .10).mean(),
                within_20pct=100 * (ape <= .20).mean())

def fit_blend(M, y_true, alphas=(0.003, 0.03, 0.3, 3.0, 30.0), seed=SEED):
    """Non-negative ridge on log1p member predictions; alpha picked by CV on the calib slice.

    Plain non-negative least squares collapses onto a single member when the members are
    highly correlated, which they always are here. A ridge penalty spreads the weight,
    which is both more stable and closer to what an ensemble is for. The penalty strength
    is chosen by 5-fold CV inside the calibration slice, scored on median APE in taka.
    """
    Z, y = np.log1p(M), np.log1p(np.asarray(y_true, dtype="float64"))
    n_splits = min(5, max(2, len(y) // 20))
    best = (np.inf, alphas[0])
    for a in alphas:
        errs = []
        for i, j in KFold(n_splits, shuffle=True, random_state=seed).split(Z):
            r = Ridge(alpha=a, positive=True, solver="lbfgs", fit_intercept=True).fit(Z[i], y[i])
            p = np.clip(np.expm1(r.predict(Z[j])), 1, None)
            t = np.clip(np.expm1(y[j]), 1, None)
            errs.append(np.median(np.abs(t - p) / t))
        score = float(np.mean(errs))
        if score < best[0]:
            best = (score, a)
    cv_medape, alpha = best
    final = Ridge(alpha=alpha, positive=True, solver="lbfgs",
                  fit_intercept=True).fit(Z, y)
    return final, alpha, cv_medape


**Read `MedAPE` and `within_20pct`, not R².** R² is dominated by the fact that long
trips cost more than short ones, which is not the part anyone needs a model for. A
driver cares how close the suggestion is in percentage terms on *their* booking, which
is what those two report.

In [ ]:
class RouteRateEncoder:
    '''Hierarchical BDT/km rate cards. Fitted per segment, on training rows only.'''

    def __init__(self, min_count=8, n_km_bands=10):
        self.min_count = min_count
        self.n_km_bands = n_km_bands

    # -- helpers ------------------------------------------------------------
    def _bands(self, km):
        return pd.Series(np.digitize(km.to_numpy(dtype="float64"), self.km_edges_),
                         index=km.index).astype(str)

    def fit(self, d, y):
        rate = (pd.Series(np.asarray(y, dtype="float64"), index=d.index)
                / d["km"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

        qs = np.linspace(0, 1, self.n_km_bands + 1)[1:-1]
        self.km_edges_ = np.unique(d["km"].quantile(qs).to_numpy())

        t = pd.DataFrame({
            "rate": rate,
            "ct":   d.car_type.astype(str),
            "pd_":  d.pickup_district.astype(str),  "dd":   d.dropoff_district.astype(str),
            "pdiv": d.pickup_division.astype(str),  "ddiv": d.dropoff_division.astype(str),
            "band": self._bands(d["km"]),
        }).dropna(subset=["rate"])

        self.global_ = float(t["rate"].median())

        def agg(keys):
            g = t.groupby(keys)["rate"].agg(["median", "size"])
            return g[g["size"] >= self.min_count]

        # district hierarchy
        self.lv1_ = agg(["pd_", "dd", "ct"])
        self.lv2_ = agg(["pd_", "dd"])
        self.lv3_ = agg(["pd_"])
        self.lv4_ = agg(["pdiv", "ddiv"])
        # distance-band hierarchy
        self.kb1_ = agg(["band", "ct"])
        self.kb2_ = agg(["band"])
        # car premium, shrunk toward 1 when a car type is rare
        cm = t.groupby("ct")["rate"].agg(["median", "size"])
        w = cm["size"] / (cm["size"] + 20)
        self.car_mult_ = (w * (cm["median"] / self.global_) + (1 - w) * 1.0)
        self.volume_ = t.groupby("pd_").size()
        return self

    def transform(self, d):
        idx = d.index
        band = self._bands(d["km"])
        mk = lambda arrs: pd.MultiIndex.from_arrays(arrs)

        def look(tab, keys):
            return (pd.Series(tab["median"].reindex(keys).to_numpy(), index=idx),
                    pd.Series(tab["size"].reindex(keys).to_numpy(), index=idx))

        ct  = d.car_type.astype(str)
        r1, n1 = look(self.lv1_, mk([d.pickup_district.astype(str),
                                     d.dropoff_district.astype(str), ct]))
        r2, n2 = look(self.lv2_, mk([d.pickup_district.astype(str),
                                     d.dropoff_district.astype(str)]))
        r3, n3 = look(self.lv3_, pd.Index(d.pickup_district.astype(str)))
        r4, n4 = look(self.lv4_, mk([d.pickup_division.astype(str),
                                     d.dropoff_division.astype(str)]))
        b1, m1 = look(self.kb1_, mk([band, ct]))
        b2, m2 = look(self.kb2_, pd.Index(band))

        route  = r1.fillna(r2).fillna(r3).fillna(r4).fillna(self.global_)
        nobs   = n1.fillna(n2).fillna(n3).fillna(n4).fillna(0)
        kmrate = b1.fillna(b2).fillna(self.global_)
        unit   = d["km"].replace(0, np.nan)

        out = pd.DataFrame(index=idx)
        out["route_ppk"]            = route.to_numpy()
        out["route_ppk_n"]          = nobs.to_numpy()
        out["baseline_fare"]        = (route * unit).to_numpy()
        out["log_baseline_fare"]    = np.log1p(out.baseline_fare.clip(lower=0))
        out["kmband_ppk"]           = kmrate.to_numpy()
        out["kmband_baseline_fare"] = (kmrate * unit).to_numpy()
        out["car_rate_mult"]        = ct.map(self.car_mult_).fillna(1.0).to_numpy()
        blend = np.sqrt(out.baseline_fare.clip(lower=1) *
                        out.kmband_baseline_fare.clip(lower=1))
        out["blend_baseline_fare"]     = blend
        out["log_blend_baseline_fare"] = np.log1p(blend)
        out["pickup_district_volume"]  = (d.pickup_district.astype(str)
                                          .map(self.volume_).fillna(0).to_numpy())
        return out


def add_route_features(fit_df, others, y_fit, n_folds=5, seed=SEED):
    '''Out-of-fold inside fit_df; a full-fit encoder applied to everything else.'''
    oof = pd.DataFrame(index=fit_df.index, columns=ROUTE_COLS, dtype=float)
    kf = KFold(n_splits=min(n_folds, max(2, len(fit_df) // 50)),
               shuffle=True, random_state=seed)
    for a, b in kf.split(fit_df):
        e = RouteRateEncoder().fit(fit_df.iloc[a], y_fit.iloc[a])
        oof.iloc[b] = e.transform(fit_df.iloc[b]).to_numpy()
    fit_df = fit_df.copy(); fit_df[ROUTE_COLS] = oof

    full = RouteRateEncoder().fit(fit_df, y_fit)
    outs = []
    for o in others:
        o = o.copy(); o[ROUTE_COLS] = full.transform(o).to_numpy(); outs.append(o)
    return fit_df, outs, full

## 13 · Design matrices and per-segment feature pruning

Numeric columns stay numeric; categoricals become pandas `category` with a shared set of
levels so LightGBM's native categorical handling works and the same encoding survives
into serving.

**Pruning** is what makes one feature list serve both segments. Inside `one_way`,
`is_round` is always 0 and the return-gap features are always missing; inside
`round_way`, `is_round` is always 1. A constant column can't be split on, but it still
occupies a slot in the feature-importance denominator and in every candidate-split scan.
Dropping it per segment is free tidiness.

In [ ]:
def prune_features(fit_df, num, cat, min_unique=2):
    '''Drop columns that are constant or entirely missing within this segment.'''
    keep_num, dropped = [], []
    for c in num:
        s = fit_df[c]
        if s.notna().sum() == 0 or s.nunique(dropna=True) < min_unique:
            dropped.append(c)
        else:
            keep_num.append(c)
    keep_cat = []
    for c in cat:
        s = fit_df[c].astype("string")
        if s.notna().sum() == 0 or s.nunique(dropna=True) < min_unique:
            dropped.append(c)
        else:
            keep_cat.append(c)
    return keep_num, keep_cat, dropped


def prep_matrix(frames, num, cat):
    '''frames[0] defines the categorical levels. Returns (matrices, dtypes, columns).'''
    cats = {}
    for c in cat:
        levels = pd.Index(frames[0][c].astype("string").fillna("NA").unique())
        cats[c] = pd.CategoricalDtype(categories=sorted(set(levels) | {"NA"}))

    out = []
    for f in frames:
        X = f.reindex(columns=num).astype("float64")
        for c in cat:
            X[c] = f[c].astype("string").fillna("NA").astype(cats[c])
        out.append(X)
    cols = list(out[0].columns)
    return [X.reindex(columns=cols) for X in out], cats, cols


def mono_vector(cols, cfg=CFG):
    '''+1 for distance-like features, 0 elsewhere. Fare must not fall as distance rises.'''
    if not cfg["MONOTONE_KM"]:
        return None
    return [1 if c in MONO_UP else 0 for c in cols]

## 14 · Optuna tuning, per segment

30 trials per segment. Each trial trains one LightGBM on the `fit` slice with early
stopping on `tune`, converts the prediction back to taka, and returns **median absolute
percentage error**. Tuning directly on the metric you care about matters here — the
default RMSE-on-log-target objective happily trades a lot of accuracy on ordinary
bookings for a little on the expensive tail.

The winning parameters are reused for every LightGBM member of the ensemble, including
the quantile models. Retuning each member separately would cost six times the runtime
for a fraction of the benefit, and would start overfitting the tune slice.



In [ ]:
BASE_PARAMS = dict(objective="regression", metric="l2", verbosity=-1,
                   seed=SEED, n_jobs=-1, feature_pre_filter=False)

DEFAULT_PARAMS = dict(learning_rate=0.04, num_leaves=64, min_data_in_leaf=30,
                      feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=1,
                      lambda_l1=0.0, lambda_l2=2.0, max_cat_threshold=64, cat_smooth=20,
                      min_gain_to_split=0.0)


def _train_lgb(params, Xa, ya, Xb, yb, wa=None, rounds=3000, stop=100,
               mono=None, quantile=None):
    p = dict(BASE_PARAMS, **params)
    if quantile is not None:
        p.update(objective="quantile", alpha=quantile, metric="quantile")
    if mono is not None:
        p.update(monotone_constraints=mono, monotone_constraints_method="advanced")
    dtrain = lgb.Dataset(Xa, ya, weight=wa, free_raw_data=False)
    dvalid = lgb.Dataset(Xb, yb, reference=dtrain, free_raw_data=False)
    return lgb.train(p, dtrain, rounds, valid_sets=[dvalid],
                     callbacks=[lgb.early_stopping(stop, verbose=False),
                                lgb.log_evaluation(0)])


def tune_segment(Xfi, Xtu, fi, tu, target="logfare", n_trials=30, seed=SEED):
    '''Returns (best_params, best_rounds, study_or_None).'''
    tgt = TARGETS[target]
    yfi, ytu = tgt["fwd"](fi), tgt["fwd"](tu)
    wfi = recency_weights(fi, CFG["RECENCY_HALFLIFE_DAYS"])
    truth = tu["fare"].to_numpy(dtype="float64")

    if n_trials <= 0:
        m = _train_lgb(DEFAULT_PARAMS, Xfi, yfi, Xtu, ytu, wa=wfi)
        return dict(DEFAULT_PARAMS), m.best_iteration, None

    def objective(trial):
        params = dict(
            learning_rate     = trial.suggest_float("learning_rate", 0.015, 0.09, log=True),
            num_leaves        = trial.suggest_int("num_leaves", 16, 256, log=True),
            min_data_in_leaf  = trial.suggest_int("min_data_in_leaf", 8, 120, log=True),
            feature_fraction  = trial.suggest_float("feature_fraction", 0.5, 1.0),
            bagging_fraction  = trial.suggest_float("bagging_fraction", 0.5, 1.0),
            bagging_freq      = 1,
            lambda_l1         = trial.suggest_float("lambda_l1", 1e-4, 5.0, log=True),
            lambda_l2         = trial.suggest_float("lambda_l2", 1e-3, 30.0, log=True),
            max_cat_threshold = trial.suggest_int("max_cat_threshold", 16, 128),
            cat_smooth        = trial.suggest_float("cat_smooth", 4.0, 60.0),
            min_gain_to_split = trial.suggest_float("min_gain_to_split", 0.0, 0.4),
        )
        m = _train_lgb(params, Xfi, yfi, Xtu, ytu, wa=wfi)
        pred = np.clip(tgt["inv"](m.predict(Xtu, num_iteration=m.best_iteration), tu), 1, None)
        trial.set_user_attr("rounds", int(m.best_iteration))
        return float(np.median(np.abs(truth - pred) / np.clip(truth, 1, None)))

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = dict(study.best_params); best["bagging_freq"] = 1
    return best, int(study.best_trial.user_attrs["rounds"]), study

## 15 · `fit_segment()` — the whole pipeline for one trip type

Everything above assembled into one function, run once per segment. It builds the
splits, fits the rate cards, prunes features, tunes, trains the ensemble members,
learns the blend weights, calibrates the band, and refits everything on all of that
segment's rows for production.

### The ensemble members

| Member | What it contributes |
|---|---|
| `lgb_fare` | tuned LightGBM on `log1p(fare)` — the workhorse |
| `lgb_ppk` | same params on `log(fare/km)` — sees rate deviation instead of scale |
| `lgb_mono` | fare constrained non-decreasing in distance — sane extrapolation |
| `lgb_resid` | predicts the gap from the blended rate-card anchor |
| `xgb` | different split algorithm, different mistakes |
| `catboost` | ordered target statistics for high-cardinality districts |

### How they're combined

Every member is converted to a **taka prediction** first, then blended with
non-negative least squares on `log1p` of those predictions, fitted on the calibration
slice. Two reasons for that shape. Working in taka space means members with different
targets are directly comparable. Forcing weights non-negative means no member can be
used as a negative correction on another — that fits the calibration noise beautifully
and generalises badly.

A member whose weight comes out at zero is telling you it adds nothing beyond the
others. That's a result, not a failure.

In [ ]:
def fit_segment(name, d_seg, cfg=CFG, verbose=True):
    log = (lambda *a: print(*a)) if verbose else (lambda *a: None)
    log(f"\n{'='*76}\n  SEGMENT: {name}   ({len(d_seg):,} rows)\n{'='*76}")

    # ---- 1. split ---------------------------------------------------------
    fi, tu, ca, te, fit_all = split_frames(d_seg, cfg)
    log(f"  fit {len(fi)} | tune {len(tu)} | calib {len(ca)} | holdout {len(te)}")
    if cfg["SPLIT_MODE"] == "time":
        log(f"  holdout window: {te.pickup_dt.min():%Y-%m-%d} -> {te.pickup_dt.max():%Y-%m-%d}")
    if len(te) < 40 or len(ca) < 40:
        log("  WARNING: very small calib/holdout — metrics and the band will be noisy.")

    # ---- 2. rate cards (fitted on the fit slice only) ---------------------
    num, cat = feature_lists(cfg)
    if cfg["USE_ROUTE_RATE"]:
        fi, (tu, ca, te), enc = add_route_features(fi, [tu, ca, te], fi["fare"])
    else:
        enc = None

    # ---- 3. prune, build matrices ----------------------------------------
    num, cat, dropped = prune_features(fi, num, cat)
    log(f"  pruned {len(dropped)} constant features: {dropped}")
    (Xfi, Xtu, Xca, Xte), cats, cols = prep_matrix([fi, tu, ca, te], num, cat)
    mono = mono_vector(cols, cfg)

    # ---- 4. tune ---------------------------------------------------------
    log(f"  tuning ({cfg['TUNE_TRIALS']} trials)...")
    params, rounds, study = tune_segment(Xfi, Xtu, fi, tu,
                                         target="logfare",
                                         n_trials=cfg["TUNE_TRIALS"], seed=SEED)
    if study is not None:
        log(f"  best tune MedAPE {100*study.best_value:.2f}%  |  rounds {rounds}")
        log(f"  params: { {k: (round(v,4) if isinstance(v,float) else v) for k,v in params.items()} }")

    wfi = recency_weights(fi, cfg["RECENCY_HALFLIFE_DAYS"])
    P = {}   # member -> dict(fit=model, pred=callable(X, frame)->fare, rounds=int)

    # ---- 5. members ------------------------------------------------------
    for tname in ("logfare", "logppk"):
        tgt = TARGETS[tname]
        m = _train_lgb(params, Xfi, tgt["fwd"](fi), Xtu, tgt["fwd"](tu), wa=wfi)
        key = "lgb_fare" if tname == "logfare" else "lgb_ppk"
        P[key] = dict(model=m, target=tname, rounds=int(m.best_iteration), kind="lgb")

    if mono is not None:
        tgt = TARGETS["logfare"]
        m = _train_lgb(params, Xfi, tgt["fwd"](fi), Xtu, tgt["fwd"](tu), wa=wfi, mono=mono)
        P["lgb_mono"] = dict(model=m, target="logfare", rounds=int(m.best_iteration),
                             kind="lgb", mono=True)

    if cfg["USE_ROUTE_RATE"]:
        anch = lambda f: np.log1p(f["blend_baseline_fare"].clip(lower=1).to_numpy())
        m = _train_lgb(params, Xfi, np.log1p(fi.fare) - anch(fi),
                       Xtu, np.log1p(tu.fare) - anch(tu), wa=wfi)
        P["lgb_resid"] = dict(model=m, target="resid", rounds=int(m.best_iteration),
                              kind="lgb_resid")

    mx = xgb.XGBRegressor(n_estimators=3000, learning_rate=params["learning_rate"],
                          max_depth=8, subsample=params["bagging_fraction"],
                          colsample_bytree=params["feature_fraction"],
                          reg_lambda=params["lambda_l2"], min_child_weight=5,
                          enable_categorical=True, tree_method="hist",
                          early_stopping_rounds=100, random_state=SEED, n_jobs=-1)
    mx.fit(Xfi, np.log1p(fi.fare), sample_weight=wfi,
           eval_set=[(Xtu, np.log1p(tu.fare))], verbose=False)
    P["xgb"] = dict(model=mx, target="logfare", kind="xgb")

    if HAS_CATBOOST:
        ci = [cols.index(c) for c in cat]
        to_cb = lambda X: X.assign(**{c: X[c].astype(str) for c in cat})
        mc = CatBoostRegressor(iterations=3000, learning_rate=params["learning_rate"],
                               depth=8, l2_leaf_reg=max(params["lambda_l2"], 1.0),
                               loss_function="RMSE", random_seed=SEED,
                               early_stopping_rounds=100, verbose=0, allow_writing_files=False)
        mc.fit(Pool(to_cb(Xfi), np.log1p(fi.fare), cat_features=ci, weight=wfi),
               eval_set=Pool(to_cb(Xtu), np.log1p(tu.fare), cat_features=ci))
        P["catboost"] = dict(model=mc, target="logfare", kind="cat",
                             cat_idx=ci, rounds=int(mc.get_best_iteration()))

    # ---- 6. member predictions -> taka -----------------------------------
    def member_fare(key, X, frame, models=None):
        e = (models or P)[key]
        m, kind = e["model"], e["kind"]
        if kind == "lgb":
            z = m.predict(X, num_iteration=getattr(m, "best_iteration", None) or e.get("rounds"))
            return np.clip(TARGETS[e["target"]]["inv"](z, frame), 1, None)
        if kind == "lgb_resid":
            z = m.predict(X, num_iteration=getattr(m, "best_iteration", None) or e.get("rounds"))
            return np.clip(np.expm1(z + np.log1p(frame["blend_baseline_fare"].clip(lower=1))), 1, None)
        if kind == "xgb":
            return np.clip(np.expm1(m.predict(X)), 1, None)
        if kind == "cat":
            Xc = X.assign(**{c: X[c].astype(str) for c in cat})
            return np.clip(np.expm1(m.predict(Pool(Xc, cat_features=e["cat_idx"]))), 1, None)
        raise ValueError(kind)

    order = list(P)
    Mca = np.column_stack([member_fare(k, Xca, ca) for k in order])
    Mte = np.column_stack([member_fare(k, Xte, te) for k in order])

    # ---- 7. non-negative ridge blend on the calibration slice -------------
    stack, alpha, cv_medape = fit_blend(Mca, ca.fare.to_numpy())
    blend = lambda M: np.clip(np.expm1(stack.predict(np.log1p(M))), 1, None)
    weights = {k: round(float(w), 4) for k, w in zip(order, stack.coef_)}
    log(f"  blend alpha {alpha} (cv MedAPE {100*cv_medape:.2f}%)  intercept {stack.intercept_:+.4f}")
    log("  blend weights:", weights)

    # ---- 8. per-member and ensemble scores on the holdout -----------------
    rows = [metrics(te.fare, Mte[:, i], k) for i, k in enumerate(order)]
    rows.append(metrics(te.fare, blend(Mte), "ENSEMBLE"))
    scores = pd.DataFrame(rows).sort_values("MedAPE").reset_index(drop=True)
    log("\n" + scores.round(3).to_string(index=False))
    top = scores.iloc[0]
    if top["model"] != "ENSEMBLE":
        ens = scores[scores.model == "ENSEMBLE"].MedAPE.iloc[0]
        log(f"  NOTE: {top['model']} alone beat the ensemble on the holdout "
            f"({top['MedAPE']:.2f}% vs {ens:.2f}%). With a holdout this size that is "
            f"usually noise, but if it persists as you add data, serve that member alone.")

    # ---- 9. conformalised band -------------------------------------------
    a = cfg["BAND_ALPHA"]
    Q = {}
    for tag, q in (("lo", a / 2), ("md", 0.5), ("hi", 1 - a / 2)):
        Q[tag] = _train_lgb(params, Xfi, np.log1p(fi.fare), Xtu, np.log1p(tu.fare),
                            wa=wfi, quantile=q)
    zlo = Q["lo"].predict(Xca); zhi = Q["hi"].predict(Xca); zt = np.log1p(ca.fare)
    scoresc = np.maximum(zlo - zt, zt - zhi)
    k = int(np.ceil((len(ca) + 1) * (1 - a))) - 1
    q_conf = float(np.sort(scoresc)[np.clip(k, 0, len(ca) - 1)])

    lo_te = np.expm1(Q["lo"].predict(Xte) - q_conf)
    hi_te = np.expm1(Q["hi"].predict(Xte) + q_conf)
    cov_raw = float(((te.fare >= np.expm1(Q["lo"].predict(Xte))) &
                     (te.fare <= np.expm1(Q["hi"].predict(Xte)))).mean())
    cov_cnf = float(((te.fare >= lo_te) & (te.fare <= hi_te)).mean())
    width   = float(np.median((hi_te - lo_te) / np.clip(blend(Mte), 1, None)))
    log(f"  band target {100*(1-a):.0f}%  |  raw {100*cov_raw:.1f}%  |  "
        f"conformal {100*cov_cnf:.1f}%  |  median width {100*width:.0f}% of the bid")

    # ---- 10. production refit on every row of the segment ----------------
    full = pd.concat([fi, tu, ca, te], ignore_index=True).sort_values("pickup_dt")
    full = full.reset_index(drop=True)
    enc_prod = RouteRateEncoder().fit(full, full["fare"]) if cfg["USE_ROUTE_RATE"] else None
    if enc_prod is not None:
        full[ROUTE_COLS] = enc_prod.transform(full).to_numpy()
    (Xfull,), _, _ = prep_matrix([full], num, cat)
    Xfull = Xfull.reindex(columns=cols)
    for c in cat:
        Xfull[c] = full[c].astype("string").fillna("NA").astype(cats[c])
    wfull = recency_weights(full, cfg["RECENCY_HALFLIFE_DAYS"])
    grow = lambda r: max(50, int(r * 1.15))

    PROD = {}
    for key, e in P.items():
        if e["kind"] in ("lgb", "lgb_resid"):
            if e["target"] == "resid":
                yv = np.log1p(full.fare) - np.log1p(full["blend_baseline_fare"].clip(lower=1))
            else:
                yv = TARGETS[e["target"]]["fwd"](full)
            p = dict(BASE_PARAMS, **params)
            if e.get("mono"):
                p.update(monotone_constraints=mono, monotone_constraints_method="advanced")
            m = lgb.train(p, lgb.Dataset(Xfull, yv, weight=wfull),
                          num_boost_round=grow(e["rounds"]))
            PROD[key] = dict(e, model=m)
        elif e["kind"] == "xgb":
            n = getattr(e["model"], "best_iteration", None) or 400
            m = xgb.XGBRegressor(n_estimators=grow(n), learning_rate=params["learning_rate"],
                                 max_depth=8, subsample=params["bagging_fraction"],
                                 colsample_bytree=params["feature_fraction"],
                                 reg_lambda=params["lambda_l2"], min_child_weight=5,
                                 enable_categorical=True, tree_method="hist",
                                 random_state=SEED, n_jobs=-1)
            m.fit(Xfull, np.log1p(full.fare), sample_weight=wfull, verbose=False)
            PROD[key] = dict(e, model=m)
        elif e["kind"] == "cat":
            m = CatBoostRegressor(iterations=grow(e["rounds"]),
                                  learning_rate=params["learning_rate"], depth=8,
                                  l2_leaf_reg=max(params["lambda_l2"], 1.0),
                                  loss_function="RMSE", random_seed=SEED, verbose=0,
                                  allow_writing_files=False)
            Xc = Xfull.assign(**{c: Xfull[c].astype(str) for c in cat})
            m.fit(Pool(Xc, np.log1p(full.fare), cat_features=e["cat_idx"], weight=wfull))
            PROD[key] = dict(e, model=m)

    QPROD = {}
    for tag, q in (("lo", a / 2), ("md", 0.5), ("hi", 1 - a / 2)):
        p = dict(BASE_PARAMS, **params, objective="quantile", alpha=q)
        QPROD[tag] = lgb.train(p, lgb.Dataset(Xfull, np.log1p(full.fare), weight=wfull),
                               num_boost_round=grow(Q[tag].best_iteration))

    return dict(
        name=name, n_rows=len(d_seg),
        num_cols=num, cat_cols=cat, fit_cols=cols, cat_dtypes=cats,
        mono=mono, params=params,
        member_order=order,
        dev_models=P, prod_models=PROD, quantile_models=QPROD,
        stack_coef=stack.coef_.tolist(), stack_intercept=float(stack.intercept_),
        weights=weights, blend_alpha=alpha, blend_cv_medape=cv_medape,
        route_encoder=enc_prod, route_encoder_dev=enc,
        q_conformal=q_conf, band_alpha=a,
        coverage_raw=cov_raw, coverage_conformal=cov_cnf, band_width=width,
        scores=scores.to_dict("records"),
        holdout_pred=blend(Mte), holdout_truth=te.fare.to_numpy(),
        holdout_frame=te, frames=dict(fit=fi, tune=tu, calib=ca, holdout=te),
        tune_best=(None if study is None else float(study.best_value)),
    )

## 16 · Fit both segments



In [ ]:
SEG = {}
for tt in CFG["SEGMENTS"]:
    d = feat[feat.trip_type == tt].reset_index(drop=True)
    if len(d) < 250:
        print(f"\nSKIPPING {tt}: only {len(d)} rows — too few for a dedicated model.")
        continue
    SEG[tt] = fit_segment(tt, d)

print("\nfitted segments:", list(SEG))

## 17 · Did any of it help?

Two questions, and they need different comparisons.

**Did segmentation help?** 

**Did the accuracy work help?** 


In [ ]:
# ---------- global baseline: one model across both segments ----------------
gnum, gcat = feature_lists(CFG)
gfi = pd.concat([s["frames"]["fit"] for s in SEG.values()], ignore_index=True)
gtu = pd.concat([s["frames"]["tune"] for s in SEG.values()], ignore_index=True)
gte = pd.concat([s["frames"]["holdout"] for s in SEG.values()], ignore_index=True)

gfi_r, (gtu_r, gte_r), _ = add_route_features(
    gfi.drop(columns=ROUTE_COLS, errors="ignore"),
    [gtu.drop(columns=ROUTE_COLS, errors="ignore"),
     gte.drop(columns=ROUTE_COLS, errors="ignore")], gfi["fare"])

gnum, gcat, _ = prune_features(gfi_r, gnum, gcat + ["trip_type"])
(GXfi, GXtu, GXte), _, gcols = prep_matrix([gfi_r, gtu_r, gte_r], gnum, gcat)

gparams, grounds, _ = tune_segment(GXfi, GXtu, gfi_r, gtu_r,
                                   n_trials=min(CFG["TUNE_TRIALS"], 15), seed=SEED)
gm = _train_lgb(gparams, GXfi, np.log1p(gfi_r.fare), GXtu, np.log1p(gtu_r.fare),
                wa=recency_weights(gfi_r, CFG["RECENCY_HALFLIFE_DAYS"]))
gte_r["global_pred"] = np.clip(np.expm1(gm.predict(GXte)), 1, None)

# ---------- side by side, per segment --------------------------------------
rows = []
for tt, s in SEG.items():
    g = gte_r[gte_r.trip_type == tt]
    rows.append(metrics(g.fare, g.global_pred, f"{tt} · global baseline"))
    rows.append(metrics(s["holdout_truth"], s["holdout_pred"], f"{tt} · SEGMENT ensemble"))
comp = pd.DataFrame(rows)
print(comp.round(3).to_string(index=False))

print("\nverdict per segment:")
for tt in SEG:
    a = comp[comp.model == f"{tt} · global baseline"].MedAPE.iloc[0]
    b = comp[comp.model == f"{tt} · SEGMENT ensemble"].MedAPE.iloc[0]
    delta = a - b
    verdict = ("segment model WINS" if delta > 0.25 else
               "global model wins — consider serving it for this segment"
               if delta < -0.25 else "too close to call")
    print(f"  {tt:11s} global {a:5.2f}%  segment {b:5.2f}%  ->  {verdict}")

In [ ]:
# ---------- where does it still go wrong? ---------------------------------
for tt, s in SEG.items():
    te = s["holdout_frame"].copy()
    te["pred"] = s["holdout_pred"]
    te["ape"] = np.abs(te.fare - te.pred) / te.fare.clip(lower=1)
    te["bias"] = te.pred - te.fare
    te["km_band"] = pd.qcut(te.km, min(5, te.km.nunique()), duplicates="drop")
    print(f"\n=== {tt} — error by distance band ===")
    print(te.groupby("km_band", observed=True).apply(lambda g: pd.Series({
        "n": len(g), "MedAPE_%": 100 * g.ape.median(),
        "within_20pct": 100 * (g.ape <= .2).mean(),
        "bias_BDT": g.bias.median(),
    }), include_groups=False).round(1).to_string())
    print(f"--- {tt} — by evidence depth ---")
    te["ev"] = pd.cut(te.route_ppk_n, [-1, 7, 24, 99, 1e9],
                      labels=["<8 (extrapolating)", "8-24 thin", "25-99 moderate", "100+ strong"])
    print(te.groupby("ev", observed=True).apply(lambda g: pd.Series({
        "n": len(g), "MedAPE_%": 100 * g.ape.median(),
    }), include_groups=False).round(1).to_string())

A positive `bias_BDT` means the model suggests **more** than drivers actually won with.
For a bid recommender that's the dangerous direction: too high loses the booking
outright, too low only costs margin. If a band shows persistent positive bias, anchor
that band on a lower quantile than P50.

The evidence table is the one to act on. If `<8 (extrapolating)` has a much worse
MedAPE than the rest — and it usually will — the app should widen or caveat the band
there rather than presenting the same confident number. It already surfaces the
evidence count for exactly this reason.

In [ ]:
# ---------- feature importance, per segment -------------------------------
fig, axes = plt.subplots(1, len(SEG), figsize=(8 * len(SEG), 7), squeeze=False)
for ax, (tt, s) in zip(axes[0], SEG.items()):
    m = s["dev_models"]["lgb_fare"]["model"]
    imp = pd.Series(m.feature_importance("gain"), index=s["fit_cols"]).sort_values()
    imp.tail(22).plot.barh(ax=ax, color="teal")
    ax.set_title(f"{tt} — gain importance (top 22)")
plt.tight_layout(); plt.show()

## 18 · The hybrid: one entry point for the app

`suggest_bid()` takes a booking keyed by **raw CSV column names**, routes on trip type
to that segment's bundle, and returns the band. The app never needs to know there are
two models behind it.

Each segment's prediction runs the same path training did — `standardize` →
`add_features` → that segment's rate-card encoder → that segment's pruned feature list
and category dtypes → its members → its blend weights → its conformal band. The one
thing to keep in mind is that this path is duplicated in the web app's
`model_service.py`;


In [ ]:
class UnsupportedTripType(ValueError):
    pass


def _prepare(seg, f):
    '''Raw frame -> design matrix, mapping unseen category levels to the NA bucket.

    A level this segment never saw in training (a district it has no trips for) would
    otherwise become NaN: LightGBM tolerates that, CatBoost raises, and either way the
    model has no information for the value. Send it to the explicit "NA" level and
    return which columns were affected so the caller can flag it.
    '''
    if seg["route_encoder"] is not None:
        f = f.copy()
        f[ROUTE_COLS] = seg["route_encoder"].transform(f).to_numpy()
    X = f.reindex(columns=seg["num_cols"]).astype("float64")
    unseen = {}
    for c in seg["cat_cols"]:
        dt = seg["cat_dtypes"][c]
        v = f[c].astype("string").fillna("NA")
        known = v.isin(list(dt.categories))
        if not known.all():
            unseen[c] = str(v[~known].iloc[0])
        X[c] = v.where(known, "NA").astype(dt)
    return f, X.reindex(columns=seg["fit_cols"]), unseen


def _member_fare(seg, key, X, f):
    e = seg["prod_models"][key]
    m, kind = e["model"], e["kind"]
    if kind == "lgb":
        return np.clip(TARGETS[e["target"]]["inv"](m.predict(X), f), 1, None)
    if kind == "lgb_resid":
        anchor = np.log1p(f["blend_baseline_fare"].clip(lower=1))
        return np.clip(np.expm1(m.predict(X) + anchor), 1, None)
    if kind == "xgb":
        return np.clip(np.expm1(m.predict(X)), 1, None)
    if kind == "cat":
        Xc = X.assign(**{c: X[c].astype(str).fillna("NA") for c in seg["cat_cols"]})
        return np.clip(np.expm1(m.predict(Pool(Xc, cat_features=e["cat_idx"]))), 1, None)
    raise ValueError(kind)


def predict_band(seg, f):
    f, X, unseen = _prepare(seg, f)
    M = np.column_stack([_member_fare(seg, k, X, f) for k in seg["member_order"]])
    point = np.clip(np.expm1(np.log1p(M) @ np.array(seg["stack_coef"])
                             + seg["stack_intercept"]), 1, None)
    q = seg["q_conformal"]
    lo = np.expm1(seg["quantile_models"]["lo"].predict(X) - q)
    hi = np.expm1(seg["quantile_models"]["hi"].predict(X) + q)
    return (point, np.clip(lo, 1, None), np.clip(hi, 1, None), f,
            dict(zip(seg["member_order"], M[0].round(0))), unseen)


def suggest_bid(booking, round_to=50):
    f = add_features(standardize(pd.DataFrame([booking])))
    tt = f["trip_type"].iloc[0]
    if tt not in SEG:
        raise UnsupportedTripType(
            f"No model for trip_type={tt!r}. Available: {sorted(SEG)}")
    if pd.isna(f["km"].iloc[0]) or f["km"].iloc[0] <= 0:
        raise ValueError("Total Km must be a positive number.")

    seg = SEG[tt]
    point, lo, hi, ff, members, unseen = predict_band(seg, f)
    rec = float(point[0])
    lo, hi = float(min(lo[0], rec)), float(max(hi[0], rec))
    rnd = lambda v: int(round(v / round_to) * round_to)
    ev = int(ff.route_ppk_n.iloc[0])
    return dict(
        segment=tt, low=rnd(lo), recommended=rnd(rec), high=rnd(hi),
        bdt_per_km=round(rec / float(ff.km.iloc[0]), 1),
        route_rate_BDT_per_km=round(float(ff.route_ppk.iloc[0]), 1),
        kmband_rate_BDT_per_km=round(float(ff.kmband_ppk.iloc[0]), 1),
        rate_card_anchor=rnd(float(ff.blend_baseline_fare.iloc[0])),
        evidence_n=ev,
        coverage_pct=int(round(100 * (1 - seg["band_alpha"]))),
        members={k: int(v) for k, v in members.items()},
        unseen_categories=unseen,
    )


demo_one_way = {
    "Trip Type": "one_way", "Car Type": "Sedan Premium", "No Of Seat": "4 Seats",
    "Total Km": "120",
    "Pickup Lat": 23.8103, "Pickup Long": 90.4125,
    "Pickup District": "Dhaka", "Pickup Division": "Dhaka",
    "Dropoff Lat": 24.3745, "Dropoff Long": 88.6042,
    "Dropoff District": "Rajshahi", "Dropoff Division": "Rajshahi",
    "Pickup Date Time": "August 05, 2026, 9:00 AM",
    "Created At": "August 04, 2026, 8:00 PM",
    "Return Date Time": None, "Driver Car Ac Status": "AC",
}
demo_round = dict(demo_one_way, **{"Trip Type": "round_way", "Total Km": "240",
                                   "Return Date Time": "August 06, 2026, 8:00 PM"})

for label, b in [("one_way 120km", demo_one_way), ("round_way 240km", demo_round)]:
    try:
        print(f"{label:18s} -> {suggest_bid(b)}")
    except UnsupportedTripType as e:
        print(f"{label:18s} -> {e}")

try:
    suggest_bid(dict(demo_one_way, **{"Trip Type": "hourly"}))
except UnsupportedTripType as e:
    print("\nrejected as expected:", e)

## 19 · Save the artifact

One file, both segments, everything the web app needs. Only production models go in —
the development models, the split frames and the Optuna studies are dropped, which
keeps the file small enough to move around.

In [ ]:
ARTIFACT = "garibook_bid_model_v3.joblib"

bundle = dict(
    version = 3,
    cfg     = CFG,
    supported_trip_types = sorted(SEG),
    car_types = sorted(clean.car_type.dropna().astype(str).unique().tolist()),
    ac_levels = sorted(clean.ac_status.dropna().astype(str).unique().tolist()),
    route_cols = ROUTE_COLS,
    comparison = comp.to_dict("records"),
    segments = {
        tt: dict(
            name            = s["name"],
            n_rows          = s["n_rows"],
            num_cols        = s["num_cols"],
            cat_cols        = s["cat_cols"],
            fit_cols        = s["fit_cols"],
            cat_dtypes      = s["cat_dtypes"],
            member_order    = s["member_order"],
            prod_models     = s["prod_models"],
            quantile_models = s["quantile_models"],
            stack_coef      = s["stack_coef"],
            stack_intercept = s["stack_intercept"],
            weights         = s["weights"],
            blend_alpha     = s["blend_alpha"],
            route_encoder   = s["route_encoder"],
            q_conformal     = s["q_conformal"],
            band_alpha      = s["band_alpha"],
            coverage_conformal = s["coverage_conformal"],
            band_width      = s["band_width"],
            params          = s["params"],
            scores          = s["scores"],
        )
        for tt, s in SEG.items()
    },
)

joblib.dump(bundle, ARTIFACT, compress=3)
import os
print(f"saved -> {ARTIFACT}  ({os.path.getsize(ARTIFACT)/1e6:.1f} MB)")
for tt, s in SEG.items():
    best = min(s["scores"], key=lambda r: r["MedAPE"])
    print(f"  {tt:11s} n={s['n_rows']:6d}  best={best['model']:10s} "
          f"MedAPE={best['MedAPE']:.2f}%  coverage={100*s['coverage_conformal']:.1f}%")

try:
    from google.colab import files as _f
    _f.download(ARTIFACT)
except Exception:
    pass